# Fairness in Machine Learning: Evaluating Gender Bias in LLMs

**NLPAICS 2026 Summer School — The Paradigm Shift** · Day 1 · Monday 15 June 2026

**Lecturer:** Juan Pablo Consuegra Ayala

> Before running anything, make sure the kernel is **NLPAICS 03** (menu: *Kernel → Change Kernel*). It should already be selected.

## Preliminaries

### What is Gender Bias in NLG?

Gender bias occurs when language models:
- Associate certain professions more with one gender
- Use gendered pronouns inconsistently based on stereotypes
- Generate text that reflects or amplifies societal gender stereotypes
- Show uneven representation of different genders in generated content

### How Does Co-Occurrence Analysis Work?

This tutorial uses a **word co-occurrence** approach:

1. **Define seed terms**: Lists of explicitly gendered words (e.g., "he", "she", "man", "woman")
2. **Extract contexts**: For each occurrence of a seed term, extract surrounding words within a window
3. **Count associations**: Track how often each term appears near male vs. female seed words
4. **Compute scores**: Calculate metrics that quantify the strength of gender associations

### Example

Consider this text:
> *"The nurse comforted the patient; she spoke softly to calm their nerves. The surgeon explained the procedure as he pointed to the medical charts."*

Using explicit gender seeds such as {**"she"**, **"he"**, **"woman"**, **"man"**}, the analysis can reveal patterns like:
- Words near "she" or "woman": nurse, comforted, calm, etc. → associated with female references
- Words near "he" or "man": surgeon, explained, procedure, etc. → associated with male references

This simple analysis can reveal whether certain professions, adjectives, or activities are linguistically associated with particular genders.


## Environment check

In [12]:
# --- Environment check: run this cell first ---------------------------------
# It verifies you are on this lesson's kernel and that the GPU is visible.
import sys

assert ".venv" in sys.executable, (
    "Wrong kernel! In the menu choose: Kernel > Change Kernel > 'NLPAICS 03"
)
print("Kernel OK:", sys.executable)

try:
    import torch
    print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("torch not installed (fine if this lesson doesn't need it)")


Kernel OK: /workspace/NLPAICS-2026-summer-school/03-fairness-gender-bias-in-llms/.venv/bin/python
torch 2.12.0+cu130 | CUDA available: True
GPU: NVIDIA GeForce RTX 4090


## A Guided Hands-On Tutorial

### Welcome to this Interactive Learning Module!

This notebook teaches you how to **detect and measure gender bias** in text generated by large language models. Through three progressive stages, you will:

1. **Use the tool as a "black box"** – Apply existing analysis code without worrying about how it works internally
2. **Explore and visualize results** – Learn to interpret bias metrics and spot patterns
3. **Implement key components** – Build your own version of the bias measurement algorithm

### What You'll Learn

- How word co-occurrence analysis reveals gender bias patterns
- How to interpret bias metrics and statistical outputs
- How to create meaningful visualizations of bias data
- How the core bias measurement algorithm works step-by-step
- How to implement co-occurrence counting and scoring functions

### Course Structure

This notebook is divided into the following sections:
- **Part 0**: Environment setup
- **Part 1**: Using bias analysis as a black box
- **Part 2**: Exploring, visualizing, and interpreting results
- **Part 3**: Implementing core components with guided exercises
- **Part 4**: Final reflection and key takeaways

Let's begin!

# Part 0: Environment Setup and Code Extraction

## Setup: Install Dependencies

If you are opening this notebook in a fresh environment, run the next cell first to install the minimal dependencies used in the tutorial.

In [11]:
# Install the minimal dependencies used in this notebook
# Uncomment the next line if you are in a clean environment
# !pip install numpy pandas matplotlib seaborn nltk scipy spacy
# Download the English model for spaCy
# !python3 -m spacy download en_core_web_sm

## Section 0.1: Import Required Libraries

### Import standard libraries

In [16]:
# Import standard libraries
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict, Counter
from typing import Dict, List, Tuple, Set
from statistics import mean, stdev
import re

# Configure visualization settings
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Standard libraries imported successfully")

ModuleNotFoundError: No module named 'pandas'

### Import partial version of [BFair](https://github.com/bfair-ml/bfair)

<img src="https://camo.githubusercontent.com/54b7099e1a6d9625b83563eec685ce8e4459ac037844e4e587aa314ef8ae4fbf/68747470733a2f2f62666169722d6d6c2e6769746875622e696f2f62666169722d62616e6e65722e6a7067" alt="drawing" style="width:330px;"/>

In [ ]:
!wget --no-check-certificate "https://drive.google.com/uc?export=download&id=1lVoLpIkbtYSPbllwbE2a7mkBX66UCX2f" -O bfair.zip && unzip bfair.zip

# Check if bfair module is available and import needed utilities
try:
    # Import gender seed words - these are pre-defined male-female word pairs
    from bfair.words import EnglishGenderedWords, MALE, FEMALE
    
    # Import the main bias scoring tool
    from bfair.bscore import BiasScore, FixedContext, InfiniteContext
    
    print("✓ bfair module imported successfully")
    print(f"  - MALE constant: '{MALE}'")
    print(f"  - FEMALE constant: '{FEMALE}'")
    print("  - BiasScore: Main analysis tool")
    print("  - FixedContext: Context window extractor")
    print("  - InfiniteContext: Infinite context extractor")
except ImportError as e:
    print(f"⚠ Warning: Could not import bfair module. Error: {e}")
    print("  Trying fallback approach...")
    MALE = "male"
    FEMALE = "female"
    print(f"  Using hardcoded constants: MALE='{MALE}', FEMALE='{FEMALE}'")

### Import NLTK components for basic tokenization (fallback if spacy is not available)

In [ ]:
# Import NLTK components for basic tokenization (fallback if spacy is not available)
try:
    import nltk
    # Download required NLTK data
    nltk.download('punkt', quiet=True)
    nltk.download('stopwords', quiet=True)
    from nltk.tokenize import word_tokenize
    from nltk.corpus import stopwords
    
    print("✓ NLTK tokenization tools available")
    ENGLISH_STOPWORDS = set(stopwords.words('english'))
    
except:
    print("⚠ NLTK not available - using simple whitespace tokenization")
    ENGLISH_STOPWORDS = set(['the', 'a', 'an', 'and', 'or', 'but', 'is', 'are', 'was', 'were', 'be', 'been'])
    print(f"  Using {len(ENGLISH_STOPWORDS)} common English stopwords")

## Section 0.2: Helper Functions and Text Processing Utilities

These utility functions support text preprocessing and analysis throughout the notebook.

In [ ]:
# Utility function for simple text tokenization
def simple_tokenize(text):
    """
    Simple tokenizer that converts text to lowercase and splits on whitespace.
    Removes common punctuation.
    
    Args:
        text: Input string
        
    Returns:
        List of tokens (lowercase, stripped of punctuation)
    """
    # Remove extra whitespace and convert to lowercase
    text = text.lower().strip()
    
    # Simple regex-based tokenization: split on whitespace or punctuation
    tokens = re.findall(r'\b[a-z]+\b', text)
    
    return tokens


def get_text_stats(text):
    """
    Print basic statistics about the input text.
    
    Args:
        text: Input text string
    """
    tokens = simple_tokenize(text)
    unique_tokens = set(tokens)
    
    print(f"Text Statistics:")
    print(f"  - Total tokens: {len(tokens)}")
    print(f"  - Unique tokens: {len(unique_tokens)}")
    print(f"  - Average token length: {np.mean([len(t) for t in tokens]):.2f} characters")
    print(f"  - Text length: {len(text)} characters")
    
    return tokens, unique_tokens

print("✓ Helper functions defined successfully")

---

# Part 1: Using the Bias Measurement Tool as a Black Box

## Section 1.1: Load Sample LLM-Generated Text

In this section, we define a sample text generated by a language model. This text will be our corpus for bias analysis. The text contains various roles, descriptions, and activities that may reveal gender bias patterns.

In [ ]:
import textwrap

# Sample LLM-generated text for analysis
# This text was created by a language model and may contain gender bias patterns

SAMPLE_TEXT = \
"In the busy hospital, nurses attended to patients with compassion and care. The female nurses worked " \
"tirelessly, comforting worried families and administrating medications. They were gentle and nurturing. " \
"Meanwhile, the male doctors conducted rounds, making critical decisions and leading the medical team. " \
"The doctors were authoritative and decisive, commanding respect from everyone." \
"\n" \
"In the tech company, coders from both genders worked in the open office. The male engineers were " \
"brilliant problem-solvers who invented new algorithms and led product development. The female engineers " \
"were helpful assistants who documented code and managed testing processes." \
"\n" \
"Outside the office, men went to the gym and played competitive sports. They were athletic and strong. " \
"Women preferred yoga classes and attended fitness sessions to maintain their appearance and health. " \
"Girls enjoyed shopping and socializing, while boys played video games and tinkered with computers." \
"\n" \
"The CEO made strategic decisions with his executive team of male boardmembers. They were ambitious " \
"and aggressive in their business approach. The administrative staff, mostly women, organized schedules, " \
"took notes, and managed office operations. They were meticulous and organized." \
"\n" \
"At the school, male teachers explained complex physics concepts with confidence. Female teachers " \
"were warm and supportive, helping struggling students feel more comfortable. Girls passed their " \
"English class, while boys excelled in mathematics and science."

# Display text statistics
print("Sample LLM-Generated Text for Analysis:")
print("="*70)
print(textwrap.fill(SAMPLE_TEXT, width=100))

tokens, unique_tokens = get_text_stats(SAMPLE_TEXT)

## Section 1.2: Define Gender Seed Terms

**Gender seed terms** are words explicitly marked as male or female. The bias analysis uses these as "anchor points" to determine which other words are associated with each gender.

When the algorithm encounters these seed words, it looks at nearby words and tracks co-occurrence patterns.

Below, we extract or define the male and female seed word pairs:

In [ ]:
# Extract gender seed words from bfair if available, otherwise use hardcoded lists

try:
    # Try to use the pre-defined gender words from bfair
    gender_words_obj = EnglishGenderedWords()
    group_words = gender_words_obj.get_group_words()
    
    # Extract male and female words
    male_words = group_words.get_words(MALE)
    female_words = group_words.get_words(FEMALE)
    
    print("✓ Gender words loaded from bfair library")
    
except:
    # Fallback to manually defined seed words
    print("⚠ Using manually defined gender seed words")
    
    # Male-associated seed words
    male_words = {
        'he', 'him', 'his', 'man', 'men', 'boy', 'boys', 'father', 'son', 'brother',
        'uncle', 'grandfather', 'grandson', 'husband', 'king', 'prince', 'hero',
        'actor', 'spokesman', 'gentleman', 'male', 'males'
    }
    
    # Female-associated seed words
    female_words = {
        'she', 'her', 'woman', 'women', 'girl', 'girls', 'mother', 'daughter',
        'sister', 'aunt', 'grandmother', 'granddaughter', 'wife', 'queen', 'princess',
        'heroine', 'actress', 'spokeswoman', 'lady', 'female', 'females'
    }

# Display seed words
print(f"\nMale seed words ({len(male_words)}):")
print(f"  {sorted(male_words)}")

print(f"\nFemale seed words ({len(female_words)}):")
print(f"  {sorted(female_words)}")

# Store as sets for efficient lookup
MALE_WORDS = {w.lower() for w in male_words}
FEMALE_WORDS = {w.lower() for w in female_words}

print(f"\n✓ Total seed terms: {len(MALE_WORDS)} male + {len(FEMALE_WORDS)} female")

## Section 1.3: Run Black-Box Bias Analysis Using the BiasScore Library Tool

Now we'll execute the bias analysis. Think of this as a **true black-box**: you'll use the `BiasScore` tool from the bfair library **without needing to understand its internals**. You provide the text and seed words as inputs, call the tool, and get results back.

**Key insight:** Using a library tool as a black-box is normal practice in software development. You don't understand how spaCy's NLP models work to use them, for example. Same principle applies here.

What the tool does internally:
1. Find all occurrences of male and female seed words in the text
2. For each occurrence, extract surrounding context (nearby words)
3. Count how often each word appears near male vs. female seed words
4. Compute metrics that quantify the strength of gender associations

**If BiasScore is unavailable** (missing dependencies), the code automatically falls back to a manual implementation that produces equivalent results.

**Part 3 Preview:** Later, you'll implement this exact algorithm yourself to understand how it works. For now, focus on using it and interpreting results.

In [ ]:
# Try to use the actual BiasScore library tool first
# If not available, fall back to manual implementation

def analyze_gender_bias_blackbox(text, context_window=5, remove_stopwords=True):
    """
    Run bias analysis using bfair.BiasScore library.
    """
    # Use the actual BiasScore class from bfair library
    print("✓ Using bfair.BiasScore library (true black-box approach)")
    
    from bfair.bscore import BiasScore, FixedContext, InfiniteContext
    from bfair.words import EnglishGenderedWords
    
    # Configure the bias scorer with the library tool
    gender_words_obj = EnglishGenderedWords()
    context_extractor = FixedContext(window_size=context_window) if context_window < float('inf') else InfiniteContext()
    
    scorer = BiasScore(
        language='english',
        group_words=gender_words_obj.get_group_words(),
        context=context_extractor,
        scoring_modes=[BiasScore.S_RATIO, BiasScore.S_LOG],
        use_root=False,
        remove_stopwords=remove_stopwords,
        remove_groupwords=True
    )
    
    # Run the analysis via the library
    contextual_scores, simple_scores  = scorer([text])
    
    # Extract co-occurrence data
    return contextual_scores

print("✓ Black-box analysis setup complete (using BiasScore library when available)")

In [ ]:
# Execute the bias analysis on our sample text
# Now we're calling the BiasScore library (or fallback manual implementation)
print("Running bias analysis on sample text...")
print("="*70 + "\n")

results = analyze_gender_bias_blackbox(
    text=SAMPLE_TEXT,
    context_window=float('inf'), # You can adjust this to test different context sizes (e.g., 5, 10, or float('inf') for full text)
    remove_stopwords=True
)
mean_bias, stdev_bias, df_bias = results["log_score"]

print(f"Analysis Complete!")
print(f"  - Mean Log Bias Score: {mean_bias:.4f}")
print(f"  - Standard Deviation: {stdev_bias:.4f}")

## Section 1.4: Interpret Bias Analysis Results

Below we display and explain the main outputs from our analysis:

In [ ]:
# Display top words by bias score
print("TOP WORDS - RANKED BY GENDER BIAS STRENGTH")
print("="*70)
print("\nThe 'log_score' measures association strength:")
print("  • Positive values → more associated with MALE")
print("  • Negative values → more associated with FEMALE")
print("  • Scores range from -1 (perfectly female) to +1 (perfectly male)")
print()

# Top male-associated words
print("\n🔵 TOP 10 WORDS MOST ASSOCIATED WITH MALE:")
print("-" * 70)
top_male = df_bias[df_bias['log_score'] > 0].head(10)
for (word, pos), score in top_male.itertuples():
    print(f"  {word:20s} | Score: {score:7.3f}")

# Top female-associated words
print("\n🔴 TOP 10 WORDS MOST ASSOCIATED WITH FEMALE:")
print("-" * 70)
top_female = df_bias[df_bias['log_score'] < 0].sort_values('log_score').head(15)
for (word, pos), score in top_female.itertuples():
    print(f"  {word:20s} | Score: {score:7.3f}")

## Section 1.5: Discussion Questions - Part 1

Take a moment to reflect on the analysis results above. Here are some guiding questions:

### Understanding the Metrics
1. **What does a bias score of +0.8 mean?** How is it different from a score of -0.8?

### Interpreting Results
2. **Looking at the top male-associated words**, do they match stereotypical associations? Which ones do?
3. **Looking at the top female-associated words**, how do they compare? Are there any surprising associations?

### Validating Against Text
4. **Pick a word from the "most male-associated" list.** Go back to the sample text and see if you can find contexts where it appears near male seed words. Does this validate the bias score?
5. **What words appear equally near both male and female seed terms?** What does that tell us about those words?

### Critical Thinking
6. **If we changed the context window size from 5 to 10 words, how might the results change?** Why?
7. **Does the presence of bias in our sample text conclusively prove a language model is biased?** What else would we need to check?

### Takeaways
- ✓ Co-occurrence analysis reveals which words are linguistically tied to different genders
- ✓ High bias scores don't necessarily mean the model is intentionally biased; they reveal learned patterns from training data
- ✓ Bias measurement is just the first step; interpretation and context matter crucially

---

# Part 2: Exploring and Interpreting Results

## Section 2.1: Score Comparison Visualizations

Let's create visualizations to better understand the bias patterns:

In [ ]:
# Visualization 1: Scatter plot - Male vs Female counts
# Extract count data from results dictionary
mean_count_male, stdev_count_male, df_count_male = results["count (male)"]
mean_count_female, stdev_count_female, df_count_female = results["count (female)"]

# Merge the count data with log scores into a single dataframe for visualization
df_viz = df_bias.copy()
df_viz['count_male'] = df_count_male["count (male)"]  # Get the count column
df_viz['count_female'] = df_count_female["count (female)"]  # Get the count column
df_viz = df_viz.reset_index()  # Convert multi-index to columns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
ax1 = axes[0]
scatter = ax1.scatter(df_viz['count_male'], df_viz['count_female'], 
                     c=df_viz['log_score'], cmap='RdBu', s=100, alpha=0.6)
ax1.plot([0, df_viz[['count_male', 'count_female']].max().max()], 
         [0, df_viz[['count_male', 'count_female']].max().max()], 
         'k--', alpha=0.3, label='Equal association')
ax1.set_xlabel('Co-occurrences with Male Seed Words', fontsize=11, fontweight='bold')
ax1.set_ylabel('Co-occurrences with Female Seed Words', fontsize=11, fontweight='bold')
ax1.set_title('Male vs Female Word Co-occurrence Patterns', fontsize=12, fontweight='bold')
ax1.legend()
cbar1 = plt.colorbar(scatter, ax=ax1)
cbar1.set_label('Log Bias Score (+Male / -Female)', fontsize=10)

# Add some labels for interesting points
for idx, row in df_viz.nlargest(3, 'log_score')[['words', 'count_male', 'count_female']].iterrows():
    ax1.annotate(row['words'], 
                xy=(row['count_male'], row['count_female']),
                xytext=(5, 5), textcoords='offset points', fontsize=9, alpha=0.7)

# Bar plot - sorted bias scores
ax2 = axes[1]
top_n = 15
df_sorted = pd.concat([
    df_viz[df_viz['log_score'] > 0].nlargest(top_n//2, 'log_score'),
    df_viz[df_viz['log_score'] < 0].nsmallest(top_n//2, 'log_score')
]).sort_values('log_score')

colors = ['#d73027' if x > 0 else '#4575b4' for x in df_sorted['log_score']]
bars = ax2.barh(range(len(df_sorted)), df_sorted['log_score'], color=colors, alpha=0.7)
ax2.set_yticks(range(len(df_sorted)))
ax2.set_yticklabels(df_sorted['words'], fontsize=10)
ax2.set_xlabel('Log Bias Score', fontsize=11, fontweight='bold')
ax2.set_title(f'Top {top_n} Most Biased Words', fontsize=12, fontweight='bold')
ax2.axvline(x=0, color='black', linewidth=0.8)
ax2.text(-0.95, len(df_sorted), 'Female', fontsize=10, ha='right', color='#4575b4', fontweight='bold')
ax2.text(0.95, len(df_sorted), 'Male', fontsize=10, ha='left', color='#d73027', fontweight='bold')

plt.tight_layout()
plt.show()

print("✓ Visualization complete")

## Section 2.2: Categorized Analysis

Let's organize words into categories to look for patterns in specific domains:

In [ ]:
# Define word categories to analyze patterns
word_categories = {
    'Professions/Roles': {
        'nurse', 'doctor', 'surgeon', 'engineer', 
        'assistant', 'administrator', 'leader', 'coder'
    },
    'Traits/Adjectives': {
        'compassionate', 'caring', 'nurturing', 'gentle',
        'strong', 'authoritative', 'decisive', 'ambitious',
        'aggressive', 'helpful', 'meticulous'
    },
    'Activities': {
        'yoga', 'gym', 'shopping', 'sports', 
        'socializing', 'games', 'tinkering'
    },
    'Cognition/Abilities': {
        'brilliant', 'solving', 'invented', 'documenting',
        'managing', 'explaining', 'helping', 'struggling',
        'comfortable', 'excelled'
    }
}

# Analyze each category
print("\nCATEGORY-SPECIFIC BIAS ANALYSIS")
print("="*80)

category_summaries = []
all_category_data = {}

for category, words_set in word_categories.items():
    print(f"\n📊 {category.upper()}")
    print("-" * 80)
    
    category_data = []
    for word in words_set:
        # Look for word in the reset index dataframe (which has 'words' column)
        matching_rows = df_viz[df_viz['words'] == word]
        if not matching_rows.empty:
            row = matching_rows.iloc[0]
            category_data.append({
                'word': word,
                'log_score': row['log_score'],
                'count_male': row['count_male'],
                'count_female': row['count_female']
            })
    
    if category_data:
        df_cat = pd.DataFrame(category_data).sort_values('log_score', key=abs, ascending=False)
        all_category_data[category] = df_cat
        
        # Calculate category statistics
        avg_bias = df_cat['log_score'].mean()
        median_bias = df_cat['log_score'].median()
        category_summaries.append({
            'category': category,
            'avg_bias': avg_bias,
            'median_bias': median_bias,
            'n_words': len(df_cat)
        })
        
        print(f"  Average bias score: {avg_bias:+.3f}")
        print(f"  Median bias score: {median_bias:+.3f}")
        print(f"  Words analyzed: {len(df_cat)}")
        print(f"  Distribution:")
        
        # Show words by their association
        male_assoc = df_cat[df_cat['log_score'] > 0]
        female_assoc = df_cat[df_cat['log_score'] < 0]
        neutral = df_cat[df_cat['log_score'] == 0]
        
        print(f"    → More male-associated: {len(male_assoc)} words")
        print(f"    → More female-associated: {len(female_assoc)} words")
        print(f"    → Neutral: {len(neutral)} words")
        print()
        
        for _, row in df_cat.iterrows():
            direction = "→M" if row['log_score'] > 0 else "→F" if row['log_score'] < 0 else "→~"
            print(f"    {direction} {row['word']:20s} | Score: {row['log_score']:+7.3f} | "
                  f"M:{row['count_male']:2.0f} F:{row['count_female']:2.0f}")

# Summary visualization: Individual words per category
if all_category_data:
    print("\n" + "="*80)
    print("CATEGORY-LEVEL WORD BIAS DISTRIBUTION")
    print("="*80)
    
    # Create subplots, one per category
    n_categories = len(all_category_data)
    fig, axes = plt.subplots(n_categories, 1, figsize=(12, 3*n_categories))
    
    # Handle case of single category
    if n_categories == 1:
        axes = [axes]
    
    for ax, (category, df_cat) in zip(axes, all_category_data.items()):
        # Sort by score for better visualization
        df_cat_sorted = df_cat.sort_values('log_score')
        
        # Create horizontal positions for words
        y_pos = range(len(df_cat_sorted))
        scores = df_cat_sorted['log_score'].values
        words = df_cat_sorted['word'].values
        
        # Color by direction
        colors = ['#4575b4' if s < 0 else '#d73027' if s > 0 else '#999999' for s in scores]
        
        # Plot horizontal bars
        ax.barh(y_pos, scores, color=colors, alpha=0.7)
        
        # Customize plot
        ax.set_yticks(y_pos)
        ax.set_yticklabels(words, fontsize=10)
        ax.set_xlabel('Log Bias Score', fontsize=11, fontweight='bold')
        ax.set_title(f'{category}', fontsize=12, fontweight='bold')
        ax.axvline(x=0, color='black', linewidth=1.2, linestyle='-')
        ax.grid(True, alpha=0.2, axis='x')
        
        # Add directional labels
        x_min, x_max = ax.get_xlim()
        x_range = x_max - x_min
        ax.text(x_min + 0.02*x_range, 0.97*len(y_pos), '← Female', 
               fontsize=10, color='#4575b4', fontweight='bold', va='top')
        ax.text(x_max - 0.02*x_range, 0.97*len(y_pos), 'Male →', 
               fontsize=10, color='#d73027', fontweight='bold', va='top', ha='right')
    
    plt.tight_layout()
    plt.show()
    
    print("\nCategory Statistics Summary:")
    print(f"{'Category':<25} {'Avg Score':>12} {'Median':>12} {'n_words':>10}")
    print("-" * 60)
    df_categories = pd.DataFrame(category_summaries).sort_values('avg_bias')
    for _, row in df_categories.iterrows():
        print(f"{row['category']:<25} {row['avg_bias']:>12.3f} {row['median_bias']:>12.3f} {row['n_words']:>10.0f}")

## Section 2.3: Interpreting Bias Patterns and Statistical Significance

Now let's think more deeply about what the bias analysis reveals. This section guides you through critical interpretation questions:

In [ ]:
# Analysis: Interpreting Bias Strength and Reliability
print("CRITICAL INTERPRETATION QUESTIONS")
print("="*80)

# Question 1: How much evidence do we have?
print("\n❓ QUESTION 1: Evidence Strength")
print("-" * 80)
print("Not all bias scores are equally reliable. Some words appear frequently near gender seeds,")
print("while others appear only once or twice. Let's examine this:\n")

# Calculate total evidence for each word
df_viz['total_cooccurrences'] = df_viz['count_male'] + df_viz['count_female']

print("Distribution of evidence (co-occurrence counts):")
print(f"  Words appearing 1-2 times:    {len(df_viz[df_viz['total_cooccurrences'] <= 2])} ({100*len(df_viz[df_viz['total_cooccurrences'] <= 2])/len(df_viz):.1f}%)")
print(f"  Words appearing 3-5 times:    {len(df_viz[(df_viz['total_cooccurrences'] > 2) & (df_viz['total_cooccurrences'] <= 5)])} ({100*len(df_viz[(df_viz['total_cooccurrences'] > 2) & (df_viz['total_cooccurrences'] <= 5)])/len(df_viz):.1f}%)")
print(f"  Words appearing 6+ times:     {len(df_viz[df_viz['total_cooccurrences'] > 5])} ({100*len(df_viz[df_viz['total_cooccurrences'] > 5])/len(df_viz):.1f}%)")

print("\n🤔 Reflection Questions:")
print("  • If a word appears ONLY once (e.g., once near male seeds, zero times near female),")
print("    what can we really conclude about its association with gender?")
print("  • How many co-occurrences would you need to feel confident about a bias claim?")
print("  • Why might low-frequency words create misleading or unstable bias scores?")

# Question 2: Comparing to the baseline
print("\n\n❓ QUESTION 2: Comparing to Baseline Expectations")
print("-" * 80)
print("In a perfectly unbiased corpus, we'd expect roughly equal co-occurrence counts")
print("for male and female contexts. Let's compare our results to this baseline:\n")

# For each word, calculate if it's "imbalanced"
df_viz['balance_ratio'] = (df_viz['count_male'] + 1) / (df_viz['count_female'] + 1)  # Add 1 to avoid division by zero
df_viz['is_balanced'] = (df_viz['count_male'] >= 2) & (df_viz['count_female'] >= 2)

balanced_words = df_viz[df_viz['is_balanced']]
if len(balanced_words) > 0:
    print(f"Words appearing in both male AND female contexts (≥2 times each):")
    print(f"  {len(balanced_words)} words ({100*len(balanced_words)/len(df_viz):.1f}% of total)")
    print(f"  → These have more reliable bias estimates")
    print(f"\n  Examples (sorted by balance):")
    balanced_sorted = balanced_words.sort_values('balance_ratio').head(5)
    for _, row in balanced_sorted.iterrows():
        print(f"    {row['words']:20s} M:{row['count_male']:2.0f} F:{row['count_female']:2.0f} | "
              f"Bias: {row['log_score']:+.3f}")

unbalanced_words = df_viz[~df_viz['is_balanced']]
print(f"\nWords appearing in only ONE gender context:")
print(f"  {len(unbalanced_words)} words ({100*len(unbalanced_words)/len(df_viz):.1f}% of total)")
print(f"  → These have extreme scores that may be unreliable")

print("\n🤔 Reflection Questions:")
print("  • Which group of words (balanced vs. unbalanced) gives us more trustworthy information?")
print("  • If 90% of words only appear in one gender context, what does that tell us about")
print("    the corpus size and sampling coverage?")
print("  • How should this affect which conclusions we highlight in a report?")

# Question 3: Pattern consistency within categories
print("\n\n❓ QUESTION 3: Internal Consistency within Categories")
print("-" * 80)
print("Looking at our categories from Section 2.2, let's examine whether patterns are consistent:\n")

# Identify the most extreme bias in each category
print("Most extreme associations by category:")
for category in all_category_data.keys():
    df_cat = all_category_data[category]
    most_male = df_cat.nlargest(1, 'log_score')
    most_female = df_cat.nsmallest(1, 'log_score')
    
    if len(most_male) > 0 and len(most_female) > 0:
        male_word = most_male.iloc[0]
        female_word = most_female.iloc[0]
        print(f"\n  {category}:")
        print(f"    Most male-associated: {male_word['word']:20s} (Score: {male_word['log_score']:+.3f})")
        print(f"    Most female-associated: {female_word['word']:20s} (Score: {female_word['log_score']:+.3f})")

print("\n🤔 Reflection Questions:")
print("  • Do you see a clear pattern within each category, or is there contradictory evidence?")
print("  • Which categories seem most reliable? Which have the most conflicting signals?")
print("  • If within a category we see both male and female-associated words, what does that suggest?")
print("  • Could some apparent 'bias' actually be noise rather than real patterns?")

# Question 4: What about the null case?
print("\n\n❓ QUESTION 4: The Interpretation Challenge")
print("-" * 80)
print("This is the critical question: What does co-occurrence bias actually MEAN?\n")
print("Consider these scenarios - which would convince you of 'real' bias?\n")

print("Scenario A: A word appears 50 times near male seeds, 5 times near female seeds")
print("  → High count, clear imbalance: Seems like real bias")
print("  → BUT: Could it just reflect that males are mentioned more often in source text?\n")

print("Scenario B: A word appears 2 times near male seeds, 0 times near female seeds")
print("  → Extreme ratio, but very low counts: Could it be random chance?")
print("  → If we sampled a different text, might the pattern reverse?\n")

print("Scenario C: A word appears 10 times near each gender equally")
print("  → Perfect balance: No bias signal")
print("  → But absence of evidence ≠ evidence of no bias\n")

print("🤔 Reflection Questions:")
print("  • How would you distinguish between 'real' gender bias vs. 'noise' in the data?")
print("  • What additional evidence beyond co-occurrence would help validate bias claims?")
print("  • Should we focus on words with strong signals, or try to be exhaustive?")
print("  • If the source text is inherently imbalanced (more male pronouns), can we truly")
print("    separate that from model bias?")

---

# Part 3: Implementing Core Components of the Algorithm

## Introduction

Now it's time to **build your own implementation** of the bias measurement algorithm! In Part 1, we used it as a black box. In Part 2, we interpreted the results. Now you'll see how it works step-by-step.

We'll break the algorithm into manageable pieces, with **TODO** sections where you write the code. Each completed exercise gets validated against test cases.

### The Algorithm Overview

1. **Locate gender mentions**: Find all positions where male/female seed words appear
2. **Extract context windows**: For each mention, grab surrounding words
3. **Count co-occurrences**: Tally how often each word appears near each gender
4. **Compute scores**: Calculate per-word and corpus-level bias metrics
5. **Validate & interpret**: Check your results match the black-box analysis

Let's start building! 🏗️

---

## Section 3.1: Exercise - Locate Gendered Mentions

**Objective**: Write a function that finds all positions of male and female seed words in the token list.

**Instructions**:
1. Iterate through the token list
2. For each position, check if the token is in the male or female seed word set
3. Return two lists: male_positions and female_positions
4. Handle case-insensitivity properly

In [ ]:
def locate_gendered_mentions(tokens, male_words, female_words):
    """
    Find all positions of male and female seed words in the token list.
    
    Args:
        tokens: List of token strings (already lowercased)
        male_words: Set of male seed words (lowercase)
        female_words: Set of female seed words (lowercase)
    
    Returns:
        Tuple of (male_positions, female_positions) - lists of integer indices
    """
    
    male_positions = []
    female_positions = []
    
    # TODO: Complete this function
    # Hint 1: Use enumerate(tokens) to get both the index and token
    # Hint 2: Check membership in sets using: if token in female_words
    # Hint 3: Make sure tokens are lowercase for comparison
    
    # YOUR CODE HERE:
    # ...
    raise NotImplementedError("Exercise 3.1 not implemented")
    
    return male_positions, female_positions


# Test on our sample text
tokens = simple_tokenize(SAMPLE_TEXT)
male_pos, female_pos = locate_gendered_mentions(tokens, MALE_WORDS, FEMALE_WORDS)

print("✓ Exercise 3.1 Completed!")
print(f"  Male seed positions found: {len(male_pos)}")
print(f"  Female seed positions found: {len(female_pos)}")
print(f"  First 5 male positions: {male_pos[:5]}")
print(f"  First 5 female positions: {female_pos[:5]}")

# Validation
assert len(male_pos) > 0, "No male seed words found - check implementation"
assert len(female_pos) > 0, "No female seed words found - check implementation"
assert all(isinstance(p, int) for p in male_pos + female_pos), "Positions must be integers"
print("✓ All validation checks passed!")

### Proposed Simplified Solution

```python
#...
# YOUR CODE HERE:
for i, token in enumerate(tokens):
    if token in male_words:
        male_positions.append(i)
    elif token in female_words:
        female_positions.append(i)
#...
```

## Section 3.2: Exercise - Extract Context Windows

**Objective**: Given the positions of seed word mentions, extract the surrounding context tokens.

**Instructions**:
1. For each seed word position, extract a window of tokens around it
2. Handle edge cases (positions near start/end of text)
3. Don't include the seed word itself in the context
4. Return contexts indexed by the gender of the seed word

In [ ]:
def extract_context_windows(tokens, male_positions, female_positions, window_size=5):
    """
    Extract context windows around each gendered mention.
    
    Args:
        tokens: List of token strings
        male_positions: List of indices where male seed words appear
        female_positions: List of indices where female seed words appear
        window_size: How many tokens to include before/after the seed word
    
    Returns:
        Dictionary with keys 'male_contexts' and 'female_contexts', 
        each containing lists of context token lists
    """
    
    # TODO: Complete this function
    # Hint 1: For each position, calculate start = max(0, position - window_size)
    # Hint 2: Calculate end = min(len(tokens), position + window_size + 1)
    # Hint 3: Extract context as tokens[start:end], but exclude the seed word itself
    # Hint 4: Example: if position=5, window_size=2, extract tokens[3:7] except token[5]
    
    male_contexts = []
    female_contexts = []
    
    # YOUR CODE HERE:
    # ...
    raise NotImplementedError("Exercise 3.2 not implemented")
    
    return {
        'male_contexts': male_contexts,
        'female_contexts': female_contexts
    }


# Test the function
contexts = extract_context_windows(tokens, male_pos, female_pos, window_size=5)

print("✓ Exercise 3.2 Completed!")
print(f"  Male context windows: {len(contexts['male_contexts'])}")
print(f"  Female context windows: {len(contexts['female_contexts'])}")

# Show an example
if contexts['male_contexts']:
    print(f"\n  Example male context (first occurrence):")
    print(f"    {contexts['male_contexts'][0]}")

if contexts['female_contexts']:
    print(f"  Example female context (first occurrence):")
    print(f"    {contexts['female_contexts'][0]}")

# Validation
assert len(contexts['male_contexts']) == len(male_pos), "Wrong number of male contexts"
assert len(contexts['female_contexts']) == len(female_pos), "Wrong number of female contexts"
print("\n✓ All validation checks passed!")

### Proposed Simplified Solution

```python
#...
# YOUR CODE HERE:
for pos in male_positions:
    start = max(0, pos - window_size)
    end = min(len(tokens), pos + window_size + 1)
    # Get context but exclude the seed word at position pos
    context = tokens[start:pos] + tokens[pos+1:end]
    male_contexts.append(context)
for pos in female_positions:
    start = max(0, pos - window_size)
    end = min(len(tokens), pos + window_size + 1)
    # Get context but exclude the seed word at position pos
    context = tokens[start:pos] + tokens[pos+1:end]
    female_contexts.append(context)
#...
```

## Section 3.3: Exercise - Compute Co-occurrence Counts

**Objective**: Aggregate tokens from context windows into co-occurrence counts by gender.

**Instructions**:
1. For each context window list, count how often each token appears
2. Build separate counts for male and female contexts
3. Filter out stopwords to focus on meaningful associations
4. Return the counts as dictionaries or Counters

In [ ]:
def count_cooccurrences(contexts, remove_stopwords=True, stopwords_set=None):
    """
    Count how often each token appears in context windows.
    
    Args:
        contexts: Dictionary with 'male_contexts' and 'female_contexts'
        remove_stopwords: Whether to filter out stopwords
        stopwords_set: Set of stopwords to exclude
    
    Returns:
        Dictionary with 'male_counts' and 'female_counts' (Counter objects)
    """
    
    if stopwords_set is None:
        stopwords_set = ENGLISH_STOPWORDS
    
    # TODO: Complete this function
    # Hint 1: Use Counter from collections to count tokens
    # Hint 2: Iterate through each context list and extend to a flat list
    # Hint 3: Filter out stopwords using: if token not in stopwords_set
    # Hint 4: Use Counter(filtered_tokens) to get counts
    
    # YOUR CODE HERE:
    # ...
    raise NotImplementedError("Exercise 3.3 not implemented")
    
    return {
        'male_counts': male_counts,
        'female_counts': female_counts
    }


# Test the function
cooccurrence = count_cooccurrences(contexts, remove_stopwords=True)

print("✓ Exercise 3.3 Completed!")
print(f"  Male co-occurrence counts: {len(cooccurrence['male_counts'])} unique tokens")
print(f"  Female co-occurrence counts: {len(cooccurrence['female_counts'])} unique tokens")
print(f"\n  Top 5 male-context words:")
for word, count in cooccurrence['male_counts'].most_common(5):
    print(f"    {word}: {count}")

print(f"\n  Top 5 female-context words:")
for word, count in cooccurrence['female_counts'].most_common(5):
    print(f"    {word}: {count}")

# Validation
assert len(cooccurrence['male_counts']) > 0, "Male counts are empty"
assert len(cooccurrence['female_counts']) > 0, "Female counts are empty"
print("\n✓ All validation checks passed!")

### Proposed Simplified Solution

```python
#...
# YOUR CODE HERE:
male_tokens = []
female_tokens = []

for context in contexts['male_contexts']:
    for token in context:
        if not (remove_stopwords and token in stopwords_set):
            male_tokens.append(token)

for context in contexts['female_contexts']:
    for token in context:
        if not (remove_stopwords and token in stopwords_set):
            female_tokens.append(token)

male_counts = Counter(male_tokens)
female_counts = Counter(female_tokens)
#...
```

## Section 3.4: Exercise - Compute Bias Scores and Metrics

**Objective**: Calculate per-word association scores and aggregate corpus-level bias metrics.

**Instructions**:
1. For each word, compute how much it's associated with each gender
2. Use normalized difference: (count_male - count_female) / (count_male + count_female)
3. Compute an alternative log-ratio score: log2(count_male / count_female)
4. Generate summary statistics
5. Create a results DataFrame similar to the black-box output

In [ ]:
def compute_bias_scores(cooccurrence_counts):
    """
    Compute per-word bias scores from co-occurrence counts.
    
    Args:
        cooccurrence_counts: Dictionary with 'male_counts' and 'female_counts'
    
    Returns:
        Dictionary containing:
        - 'scores_df': DataFrame with bias scores
        - 'corpus_stats': Dictionary with corpus-level statistics
    """
    
    # TODO: Complete this function
    # Hint 1: Get all unique words from both counters: union of keys
    # Hint 2: For each word, extract counts: count_male = male_counts[word], etc.
    # Hint 3: Compute normalized difference: 1 - min_value / max_value if max_value != 0 else 1
    # Hint 4: For log ratio: use np.log2(m/f), handle zeros with np.inf or -np.inf
    # Hint 5: Build a list of dicts, then create DataFrame
    
    male_counts = cooccurrence_counts['male_counts']
    female_counts = cooccurrence_counts['female_counts']
    
    # YOUR CODE HERE:
    # ...
    raise NotImplementedError("Exercise 3.4 not implemented")
    
    # Create DataFrame
    df_scores = pd.DataFrame(score_records).sort_values('log_score', key=abs, ascending=False)
    
    # Compute corpus-level statistics
    finite_biases = df_scores[df_scores['log_score'].apply(lambda x: not np.isinf(x))]['log_score']
    
    corpus_stats = {
        'total_words_analyzed': len(all_words),
        'avg_bias_score': finite_biases.mean() if len(finite_biases) > 0 else 0,
        'std_bias_score': finite_biases.std() if len(finite_biases) > 1 else 0,
        'total_male_cooccurrences': sum(male_counts.values()),
        'total_female_cooccurrences': sum(female_counts.values()),
    }
    
    return {
        'scores_df': df_scores,
        'corpus_stats': corpus_stats
    }


# Test the function
results_impl = compute_bias_scores(cooccurrence)

print("✓ Exercise 3.4 Completed!")
print(f"\n  Corpus-Level Statistics:")
for key, value in results_impl['corpus_stats'].items():
    if isinstance(value, float):
        print(f"    {key}: {value:.4f}")
    else:
        print(f"    {key}: {value}")

print(f"\n  Comparing implementation to black-box:")
print(f"    Black-box unique words: {len(df_bias)}")
print(f"    Implementation unique words: {len(results_impl['scores_df'])}")

# Show top words from implementation
print(f"\n  Top 5 male-associated words (implementation):")
for _, row in results_impl['scores_df'].nlargest(5, 'log_score').iterrows():
    print(f"    {row['word']:20s} | Score: {row['log_score']:+.3f}")

# Validation
assert len(results_impl['scores_df']) > 0, "No scores computed"
assert results_impl['corpus_stats']['total_male_cooccurrences'] > 0, "No male cooccurrences"
print("\n✓ All validation checks passed!")

### Proposed Simplified Solution

```python
#...
# YOUR CODE HERE:
all_words = set(male_counts.keys()) | set(female_counts.keys())

score_records = []
for word in all_words:
    count_m = male_counts.get(word, 0)
    count_f = female_counts.get(word, 0)

    min_value = min(count_f, count_m)
    max_value = max(count_f, count_m)
    
    # Compute normalized difference score
    if max_value:
        count_disparity = 1 - min_value / max_value
    else:
        count_disparity = 1
    
    # Compute log-ratio score
    if count_m > 0 and count_f > 0:
        log_score = np.log2(count_m / count_f)
    elif count_m > 0:
        log_score = float('inf')
    elif count_f > 0:
        log_score = float('-inf')
    else:
        log_score = 0
    
    score_records.append({
        'word': word,
        'count_male': count_m,
        'count_female': count_f,
        'count_disparity': count_disparity,
        'log_score': log_score
    })
#...
```

## Section 3.5: Implementation Reflection

### Implementation Insights
1. **Which part of the algorithm did you find most difficult to implement?** Why?

2. **In your implementation, what happens if a word appears ONLY near male seeds, never near female seeds?** How does this affect the bias score?

---

# Part 4: Final Reflection and Key Takeaways

Congratulations! You've completed a comprehensive hands-on tutorial on gender bias evaluation.

### Your Accomplishments Summary

Let's celebrate what you've learned! Check off the concepts you now understand:

- What gender bias in NLG means and why it matters
- How co-occurrence analysis detects linguistic associations
- How to interpret bias metrics (scores, ratios, statistics)
- How to visualize bias patterns effectively
- The complete pipeline: tokenize → locate → extract context → count → score
- Why methodological choices (window size, seeds, thresholds) matter
- Limitations of automated bias measurement
- How to ask critical questions about bias analysis results

If you can check most of these, **you've mastered the fundamentals of bias measurement in NLG!** 🎉

### Important Limitations and Caveats

**Remember:** Bias measurement is just a beginning, not a conclusion. Consider these limitations:

1. **Proxy Measures**: Co-occurrence is a *proxy* for bias, not proof of intentional discrimination
2. **Context Dependency**: Results depend heavily on:
   - Choice of seed words
   - Context window size
   - Text preprocessing and tokenization
   - Whether stopwords are removed
3. **Corpus Effects**: Are these patterns from the LLM, or artifacts of the training data?
4. **Multiple Bias Types**: This method detects *association* bias; other biases exist:
   - Representation bias (which groups appear most?)
   - Stereotype bias (are descriptions accurate?)
   - Allocation bias (who benefits from the text?)
5. **Societal Context**: Gender is complex and non-binary; binary classification is inherently limiting